In [1]:
# =========================
# LOAD LIBRARIES
# =========================

import pandas as pd
import numpy as np

In [2]:
# =========================
# LOAD MATCH RESULTS
# =========================

df = pd.read_csv("../data/raw/results.csv")

df["date"] = pd.to_datetime(df["date"])

df = df.dropna(subset=["home_score", "away_score"]).copy()

df["home_score"] = df["home_score"].astype(int)
df["away_score"] = df["away_score"].astype(int)

df = df[df["date"].dt.year >= 2006].copy()

df = df.sort_values("date").reset_index(drop=True)

print("Shape:", df.shape)
print("Date range:", df["date"].min(), "->", df["date"].max())

df.head()

Shape: (19488, 9)
Date range: 2006-01-02 00:00:00 -> 2026-03-31 00:00:00


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,2006-01-02,Qatar,Libya,2,0,Friendly,Doha,Qatar,False
1,2006-01-05,Egypt,Zimbabwe,2,0,Friendly,Alexandria,Egypt,False
2,2006-01-07,Guinea,Togo,1,0,Friendly,Viry-Châtillon,France,True
3,2006-01-09,Morocco,DR Congo,3,0,Friendly,Rabat,Morocco,False
4,2006-01-11,Ghana,Togo,0,1,Friendly,Monastir,Tunisia,True


In [3]:
# =========================
# BASIC TARGET COLUMNS
# =========================

df["home_result"] = np.where(
    df["home_score"] > df["away_score"], 1,
    np.where(df["home_score"] == df["away_score"], 0, -1)
)

df["away_result"] = -df["home_result"]

df["home_points"] = df["home_result"].map({
    1: 3,
    0: 1,
    -1: 0
})

df["away_points"] = df["away_result"].map({
    1: 3,
    0: 1,
    -1: 0
})

df["total_goals"] = df["home_score"] + df["away_score"]
df["goal_diff"] = df["home_score"] - df["away_score"]

df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_result,away_result,home_points,away_points,total_goals,goal_diff
0,2006-01-02,Qatar,Libya,2,0,Friendly,Doha,Qatar,False,1,-1,3,0,2,2
1,2006-01-05,Egypt,Zimbabwe,2,0,Friendly,Alexandria,Egypt,False,1,-1,3,0,2,2
2,2006-01-07,Guinea,Togo,1,0,Friendly,Viry-Châtillon,France,True,1,-1,3,0,1,1
3,2006-01-09,Morocco,DR Congo,3,0,Friendly,Rabat,Morocco,False,1,-1,3,0,3,3
4,2006-01-11,Ghana,Togo,0,1,Friendly,Monastir,Tunisia,True,-1,1,0,3,1,-1


In [4]:
# =========================
# FORM FEATURES FUNCTION
# =========================

def calculate_form_features(
    df,
    team_col,
    points_col,
    goals_scored_col,
    goals_conceded_col,
    window=5
):
    temp = pd.DataFrame()

    temp["team"] = df[team_col]
    temp["date"] = df["date"]
    temp["points"] = df[points_col]
    temp["goals_scored"] = df[goals_scored_col]
    temp["goals_conceded"] = df[goals_conceded_col]

    temp = temp.sort_values("date")

    temp["form_points"] = (
        temp.groupby("team")["points"]
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )

    temp["form_goals_scored"] = (
        temp.groupby("team")["goals_scored"]
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )

    temp["form_goals_conceded"] = (
        temp.groupby("team")["goals_conceded"]
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )

    return temp[[
        "form_points",
        "form_goals_scored",
        "form_goals_conceded"
    ]]

In [5]:
# =========================
# CREATE FORM FEATURES
# =========================

home_form = calculate_form_features(
    df,
    "home_team",
    "home_points",
    "home_score",
    "away_score"
)

home_form.columns = [
    "home_form_points",
    "home_form_goals_scored",
    "home_form_goals_conceded"
]

away_form = calculate_form_features(
    df,
    "away_team",
    "away_points",
    "away_score",
    "home_score"
)

away_form.columns = [
    "away_form_points",
    "away_form_goals_scored",
    "away_form_goals_conceded"
]

df = pd.concat([df, home_form, away_form], axis=1)

df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_result,...,home_points,away_points,total_goals,goal_diff,home_form_points,home_form_goals_scored,home_form_goals_conceded,away_form_points,away_form_goals_scored,away_form_goals_conceded
0,2006-01-02,Qatar,Libya,2,0,Friendly,Doha,Qatar,False,1,...,3,0,2,2,NaN,NaN,NaN,NaN,NaN,NaN
1,2006-01-05,Egypt,Zimbabwe,2,0,Friendly,Alexandria,Egypt,False,1,...,3,0,2,2,NaN,NaN,NaN,NaN,NaN,NaN
2,2006-01-07,Guinea,Togo,1,0,Friendly,Viry-Châtillon,France,True,1,...,3,0,1,1,NaN,NaN,NaN,NaN,NaN,NaN
3,2006-01-09,Morocco,DR Congo,3,0,Friendly,Rabat,Morocco,False,1,...,3,0,3,3,NaN,NaN,NaN,NaN,NaN,NaN
4,2006-01-11,Ghana,Togo,0,1,Friendly,Monastir,Tunisia,True,-1,...,0,3,1,-1,NaN,NaN,NaN,0.0,0.0,1.0


In [6]:
# =========================
# TOURNAMENT WEIGHTS
# =========================

def get_tournament_weight(tournament):
    tournament = tournament.lower()

    if tournament == "fifa world cup":
        return 5

    if any(x in tournament for x in [
        "uefa euro",
        "copa américa",
        "african cup",
        "asian cup",
        "gold cup",
        "nations league"
    ]):
        return 4

    if "qualification" in tournament:
        return 3

    if "friendly" in tournament:
        return 1

    return 2


df["tournament_weight"] = df["tournament"].apply(get_tournament_weight)

df["is_world_cup"] = (df["tournament"] == "FIFA World Cup").astype(int)
df["is_qualifier"] = df["tournament"].str.contains("qualification", case=False, na=False).astype(int)
df["is_friendly"] = (df["tournament"] == "Friendly").astype(int)

df[[
    "tournament",
    "tournament_weight",
    "is_world_cup",
    "is_qualifier",
    "is_friendly"
]].drop_duplicates().head(20)

,tournament,tournament_weight,is_world_cup,is_qualifier,is_friendly
0,Friendly,1,0,0,1
14,African Cup of Nations,4,0,0,0
44,Lunar New Year Cup,2,0,0,0
81,AFC Asian Cup qualification,4,0,1,0
99,Cyprus International Tournament,2,0,0,0
133,Malta International Tournament,2,0,0,0
145,Muratti Vase,2,0,0,0
153,AFC Challenge Cup,2,0,0,0
187,COSAFA Cup,2,0,0,0
195,Kirin Cup,2,0,0,0


In [7]:
# =========================
# LOAD + CLEAN FIFA RANKINGS
# =========================

rankings = pd.read_csv("../data/raw/fifa_mens_rank.csv")

# Create real ranking dates from year + semester
rankings["ranking_date"] = pd.to_datetime(
    rankings["date"].astype(str) + "-" +
    rankings["semester"].map({
        1: "01-01",
        2: "07-01"
    })
)

# Keep only relevant columns
rankings = rankings.rename(columns={
    "rank": "fifa_rank",
    "total.points": "fifa_points"
})

rankings = rankings[[
    "ranking_date",
    "team",
    "fifa_rank",
    "fifa_points"
]].copy()

# Keep rankings only from 2006+
rankings = rankings[
    rankings["ranking_date"].dt.year >= 2006
].copy()

# Sort
rankings = rankings.sort_values(
    ["team", "ranking_date"]
).reset_index(drop=True)

print("Rankings shape:", rankings.shape)

rankings.head()

Rankings shape: (7945, 4)


,ranking_date,team,fifa_rank,fifa_points
0,2006-01-01,Afghanistan,174,48.0
1,2006-07-01,Afghanistan,179,39.0
2,2007-01-01,Afghanistan,188,18.0
3,2007-07-01,Afghanistan,192,16.0
4,2008-01-01,Afghanistan,181,53.0


In [8]:
# =========================
# FIX TEAM NAME MISMATCHES
# =========================

team_name_mapping = {

    # Main mismatches
    "United States": "USA",
    "South Korea": "Korea Republic",
    "Iran": "IR Iran",
    "Ivory Coast": "Côte d'Ivoire",
    "DR Congo": "Congo DR",
    "North Macedonia": "Macedonia FYR",
    "North Korea": "Korea DPR",

    # Caribbean
    "Saint Kitts and Nevis": "St. Kitts and Nevis",
    "Saint Lucia": "St. Lucia",
    "Saint Vincent and the Grenadines": "St. Vincent / Grenadines",

    # Africa / Asia
    "Cabo Verde": "Cape Verde",
    "Kyrgyzstan": "Kyrgyz Republic",
    "Brunei": "Brunei Darussalam",

    # Unicode
    "Curaçao": "Curacao",

    # Political naming
    "Taiwan": "Chinese Taipei"
}

# Save original names
df["home_team_original"] = df["home_team"]
df["away_team_original"] = df["away_team"]

# Apply fixes
df["home_team"] = df["home_team"].replace(team_name_mapping)
df["away_team"] = df["away_team"].replace(team_name_mapping)

df[["home_team_original", "home_team"]].head(20)

,home_team_original,home_team
0,Qatar,Qatar
1,Egypt,Egypt
2,Guinea,Guinea
3,Morocco,Morocco
4,Ghana,Ghana
5,Tunisia,Tunisia
6,Morocco,Morocco
7,Senegal,Senegal
8,Egypt,Egypt
9,Tunisia,Tunisia


In [9]:
# =========================
# MERGE FIFA RANKINGS
# =========================

# Sort before merge
df = df.sort_values("date").reset_index(drop=True)

# ---------- HOME TEAM ----------

home_rankings = rankings.rename(columns={
    "team": "home_team",
    "fifa_rank": "home_fifa_rank",
    "fifa_points": "home_fifa_points"
})

home_rankings = home_rankings.sort_values(
    "ranking_date"
).reset_index(drop=True)

df = pd.merge_asof(
    df.sort_values("date"),
    home_rankings,
    left_on="date",
    right_on="ranking_date",
    by="home_team",
    direction="backward"
)

df = df.rename(columns={
    "ranking_date": "home_ranking_date"
})

# ---------- AWAY TEAM ----------

away_rankings = rankings.rename(columns={
    "team": "away_team",
    "fifa_rank": "away_fifa_rank",
    "fifa_points": "away_fifa_points"
})

away_rankings = away_rankings.sort_values(
    "ranking_date"
).reset_index(drop=True)

df = pd.merge_asof(
    df.sort_values("date"),
    away_rankings,
    left_on="date",
    right_on="ranking_date",
    by="away_team",
    direction="backward"
)

df = df.rename(columns={
    "ranking_date": "away_ranking_date"
})

# Final sort
df = df.sort_values("date").reset_index(drop=True)

print(df.shape)

df.head()

(19488, 33)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_result,...,is_qualifier,is_friendly,home_team_original,away_team_original,home_ranking_date,home_fifa_rank,home_fifa_points,away_ranking_date,away_fifa_rank,away_fifa_points
0,2006-01-02,Qatar,Libya,2,0,Friendly,Doha,Qatar,False,1,...,0,1,Qatar,Libya,2006-01-01,77.0,420.0,2006-01-01,80.0,401.0
1,2006-01-05,Egypt,Zimbabwe,2,0,Friendly,Alexandria,Egypt,False,1,...,0,1,Egypt,Zimbabwe,2006-01-01,29.0,819.0,2006-01-01,67.0,476.0
2,2006-01-07,Guinea,Togo,1,0,Friendly,Viry-Châtillon,France,True,1,...,0,1,Guinea,Togo,2006-01-01,24.0,850.0,2006-01-01,49.0,622.0
3,2006-01-09,Morocco,Congo DR,3,0,Friendly,Rabat,Morocco,False,1,...,0,1,Morocco,DR Congo,2006-01-01,41.0,681.0,2006-01-01,68.0,475.0
4,2006-01-11,Ghana,Togo,0,1,Friendly,Monastir,Tunisia,True,-1,...,0,1,Ghana,Togo,2006-01-01,25.0,839.0,2006-01-01,49.0,622.0


In [17]:
# =========================
# CHECK FIFA RESULTS
# =========================

print(
    "Missing home FIFA rank:",
    df["home_fifa_rank"].isnull().sum()
)

print(
    "Missing away FIFA rank:",
    df["away_fifa_rank"].isnull().sum()
)

df[[
    "date",
    "home_team",
    "away_team",
    "home_fifa_rank",
    "away_fifa_rank",
    "home_fifa_points",
    "away_fifa_points"
]].head(20)

Missing home FIFA rank: 1173
Missing away FIFA rank: 1197


,date,home_team,away_team,home_fifa_rank,away_fifa_rank,home_fifa_points,away_fifa_points
0,2006-01-02,Qatar,Libya,77.0,80.0,420.0,401.0
1,2006-01-05,Egypt,Zimbabwe,29.0,67.0,819.0,476.0
2,2006-01-07,Guinea,Togo,24.0,49.0,850.0,622.0
3,2006-01-09,Morocco,DR Congo,41.0,68.0,681.0,475.0
4,2006-01-11,Ghana,Togo,25.0,49.0,839.0,622.0
5,2006-01-12,Tunisia,Libya,31.0,80.0,805.0,401.0
6,2006-01-14,Senegal,DR Congo,35.0,68.0,726.0,475.0
7,2006-01-14,Egypt,South Africa,29.0,73.0,819.0,455.0
8,2006-01-14,Morocco,Zimbabwe,41.0,67.0,681.0,476.0
9,2006-01-15,Tunisia,Ghana,31.0,25.0,805.0,839.0


In [18]:
# =========================
# FIND UNMATCHED TEAMS
# =========================

missing_home = df[df["home_fifa_rank"].isnull()]["home_team"].value_counts()
missing_away = df[df["away_fifa_rank"].isnull()]["away_team"].value_counts()

missing_teams = pd.concat([missing_home, missing_away], axis=1).fillna(0)
missing_teams.columns = ["missing_home_count", "missing_away_count"]
missing_teams["total_missing"] = (
    missing_teams["missing_home_count"] + missing_teams["missing_away_count"]
)

missing_teams = missing_teams.sort_values("total_missing", ascending=False)

missing_teams.head(50)

,missing_home_count,missing_away_count,total_missing
North Macedonia,97.0,96.0,193.0
Cabo Verde,68.0,81.0,149.0
Guadeloupe,63.0,60.0,123.0
Martinique,58.0,61.0,119.0
French Guiana,47.0,51.0,98.0
Jersey,30.0,31.0,61.0
Zanzibar,24.0,36.0,60.0
United States Virgin Islands,27.0,30.0,57.0
Guernsey,24.0,29.0,53.0
Saint Martin,18.0,34.0,52.0


In [19]:
ranking_teams = set(rankings["team"].unique())
match_teams = set(df["home_team"].unique()).union(set(df["away_team"].unique()))

unmatched = sorted(match_teams - ranking_teams)

print("Number of unmatched teams:", len(unmatched))
unmatched[:100]

Number of unmatched teams: 114


['Abkhazia',
 'Alderney',
 'Ambazonia',
 'Andalusia',
 'Arameans Suryoye',
 'Artsakh',
 'Aymara',
 'Barawa',
 'Basque Country',
 'Biafra',
 'Bonaire',
 'Brittany',
 'Canary Islands',
 'Cascadia',
 'Catalonia',
 'Chagos Islands',
 'Chameria',
 'Chechnya',
 'Cilento',
 'Corsica',
 'County of Nice',
 'Crimea',
 'DR Congo',
 'Darfur',
 'Donetsk PR',
 'Délvidék',
 'Elba Island',
 'Ellan Vannin',
 'Falkland Islands',
 'Felvidék',
 'Franconia',
 'French Guiana',
 'Frøya',
 'Galicia',
 'Gotland',
 'Gozo',
 'Greenland',
 'Guadeloupe',
 'Guernsey',
 'Găgăuzia',
 'Hitra',
 'Hmong',
 'Iran',
 'Iraqi Kurdistan',
 'Isle of Man',
 'Isle of Wight',
 'Ivory Coast',
 'Jersey',
 'Kabylia',
 'Kernow',
 'Kiribati',
 'Kyrgyzstan',
 'Kárpátalja',
 'Luhansk PR',
 'Madrid',
 'Mapuche',
 'Marshall Islands',
 'Martinique',
 'Matabeleland',
 'Maule Sur',
 'Mayotte',
 'Menorca',
 'Monaco',
 'Northern Cyprus',
 'Northern Mariana Islands',
 'Occitania',
 'Orkney',
 'Padania',
 'Panjab',
 'Parishes of Jersey',
 'Prov